# วิเคราะห์ความสำคัญของถนน (Road Network Centrality & Traffic Proxy)
สมุดโน้ตเล่มนี้ออกแบบมาเพื่อวิเคราะห์โครงข่ายถนนในกรุงเทพฯ เพื่อประเมิน **ปริมาณคนเดินและรถยนต์สัญจร (Traffic/Foot Traffic Proxy)** โดยใช้วิธีทางวิทยาศาสตร์ข้อมูลเชิงเครือข่าย (Network Science) สองแนวทาง:
1. **Road Class Weighting (เร็วและรองรับขนาดใหญ่):** การกำหนดค่าน้ำหนักความสำคัญตามประเภทของถนน (เช่น ถนนเส้นหลัก > ถนนซอย)
2. **Graph Centrality (วิเคราะห์เชิงโครงสร้างในพื้นที่เป้าหมาย):** คำนวณความใกล้ชิดศูนย์กลาง (Closeness Centrality) บนจุดตัดเครือข่ายถนนจริงเพื่อหาแยกที่มีความต้องการสัญจรสูงสุด

In [ ]:
import pandas as pd
import geopandas as gpd
from pyrosm import OSM
import networkx as nx
import os
import gc
from pathlib import Path

# ค้นหาตำแหน่งโฟลเดอร์หลักของโปรเจกต์
NOTEBOOK_DIR = Path(os.getcwd())
BASE_DIR = NOTEBOOK_DIR.parent.parent # เลื่อนขึ้นไปที่ ZoneVision/data-pipeline

print(f"Notebook Directory: {NOTEBOOK_DIR}")
print(f"Base Directory: {BASE_DIR}")

In [ ]:
# โหลดขอบเขตแผนที่กรุงเทพฯ
import json
config_path = BASE_DIR / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

osm_filepath = str(BASE_DIR / "data" / "raw" / "osm.pbf")
bkk_bbox = config.get("bkk_bbox", [100.30, 13.45, 100.95, 13.95])

print(f"OSM Path: {osm_filepath}")

In [ ]:
# 1. ดึงเครือข่ายถนนรถวิ่ง (Driving Network)
print("กำลังเชื่อมต่อและดึงข้อมูลเส้นทางถนนกรุงเทพฯ (อาจใช้เวลาประมาณ 1-2 นาทีเนื่องจากข้อมูลถนนมีปริมาณมาก)...")
osm = OSM(osm_filepath, bounding_box=bkk_bbox)
drive_net = osm.get_network(network_type="driving")

print(f"ดึงข้อมูลเส้นขอบถนนดิบมาได้: {len(drive_net)} เส้น")
print(drive_net[['name', 'highway', 'geometry']].head(5))

In [ ]:
# แนวทางที่ 1: Road Class Weighting (วิธีที่เร็วและเสถียรที่สุดสำหรับทั้งจังหวัด)
# กำหนดค่าน้ำหนักปริมาณรถวิ่งโดยเฉลี่ยตามความสำคัญของประเภทถนน

road_weights = {
    'motorway': 10,        # ทางด่วน/มอเตอร์เวย์
    'trunk': 9,           # ถนนเลี่ยงเมืองเส้นหลัก
    'primary': 8,         # ถนนใหญ่เชื่อมเมือง (เช่น ถนนสุขุมวิท, ถนนวิภาวดี)
    'secondary': 6,       # ถนนหลักรองลงมา (เช่น ถนนเอกมัย, ถนนทองหล่อ)
    'tertiary': 4,        # ถนนเชื่อมย่อย (เช่น ถนนในซอยใหญ่เชื่อมสองถนนหลัก)
    'residential': 2,     # ถนนในหมู่บ้าน/ซอยบ้านพักอาศัย
    'living_street': 1    # ซอยแคบมาก/ถนนแชร์ความปลอดภัย
}

print("กำลังคำนวณค่าน้ำหนักถนนตามระดับชั้นการจราจร...")
drive_net['traffic_weight'] = drive_net['highway'].map(road_weights).fillna(1.0)

# สกัดจุดกึ่งกลางของเส้นถนนมาเป็นตัวแทนพิกัดในการคำนวณทำเลทองย่อย
drive_net['latitude'] = drive_net.geometry.centroid.y
drive_net['longitude'] = drive_net.geometry.centroid.x

road_traffic_proxy = pd.DataFrame(drive_net[['name', 'highway', 'traffic_weight', 'latitude', 'longitude']])
road_traffic_proxy = road_traffic_proxy.dropna(subset=['name'])

print(f"สร้างดัชนีคะแนนการจราจรถนิมสำเร็จคงเหลือถนนมีชื่อ: {len(road_traffic_proxy)} เส้น")
print(road_traffic_proxy.sort_values(by='traffic_weight', ascending=False).head(10))

In [ ]:
# แนวทางที่ 2: Graph Centrality (วิเคราะห์เฉพาะพื้นที่เพื่อหลีกเลี่ยงหน่วยความจำแรมเต็ม)
# ⚠️ คำเตือน: ห้ามคำนวณ Centrality ของทั้งจังหวัดกรุงเทพฯ พร้อมกันบนเครื่องปกติเพราะแรมจะล้นและระบบจะค้าง
# เราจะทำการสร้างกราฟย่อยรอบพิกัดตัวอย่าง (เช่น บริเวณย่านสยามสแควร์)

sample_lat = 13.7456
sample_lng = 100.5342
radius_deg = 0.015 # รัศมีรอบจุดวิเคราะห์ประมาณ 1.5 - 2 กิโลเมตร

print(f"กำลังตัดข้อมูลเครือข่ายถนนย่อยรอบพิกัดทดสอบ (Siam Area: {sample_lat}, {sample_lng})...")
local_net = drive_net[
    (drive_net['latitude'].between(sample_lat - radius_deg, sample_lat + radius_deg)) &
    (drive_net['longitude'].between(sample_lng - radius_deg, sample_lng + radius_deg))
].copy()

print(f"ดึงถนนรอบพื้นที่เป้าหมายมาได้: {len(local_net)} เส้น")

In [ ]:
# แปลงโครงข่ายแผนที่ถนนย่อยให้กลายเป็นโครงสร้างข้อมูลกราฟ (Graph)
print("กำลังแปลงข้อมูลเชิงพื้นที่ถนนเป็นกราฟเครือข่ายด้วย pyrosm...")
try:
    # แปลงเป็น NetworkX Graph สำหรับรันคณิตศาสตร์เชิงเครือข่าย
    nodes, edges = osm.to_graph(local_net, graph_type="networkx", retain_all=True)
    
    # คำนวณหาค่าจุดตัดถนนที่มีผู้คนสัญจรผ่านบ่อยที่สุด (Closeness Centrality)
    # Closeness Centrality จะช่วยชี้ว่าแยกไหนเข้าถึงง่ายและเป็นศูนย์รวมเส้นทางเดินรถย่านนั้น
    print("กำลังคำนวณหาค่าความใกล้ชิดศูนย์กลาง (Closeness Centrality) ของแต่ละจุดตัดทางแยก...")
    centrality = nx.closeness_centrality(nodes)
    
    # นำค่าน้ำหนักส่งกลับคืนตารางพิกัดจุดแยกเพื่อจัดอันดับพิกัดทองคำ
    nx.set_node_attributes(nodes, centrality, "closeness")
    
    # แปลง Nodes ในระบบกราฟกลับมาดูสถิติแยกทองคำ
    df_nodes = pd.DataFrame.from_dict(dict(nodes.nodes(data=True)), orient='index')
    
    print("🎉 ค้นพบแยกทองคำที่มีการเข้าถึงเชื่อมต่อในพื้นที่ดีที่สุด 5 อันดับแรก:")
    print(df_nodes.sort_values(by='closeness', ascending=False)[['y', 'x', 'closeness']].head(5))
except Exception as e:
    print(f"⚠️ เกิดข้อผิดพลาดในการรันกราฟ: {str(e)}")
    print("ข้อเสนอแนะ: ในการนำไปทำจริง แนะนำให้ใช้แนวทางที่ 1 (Road Class Weighting) ในการสเกลข้อมูลแผนที่ใหญ่จะเสถียรที่สุด")

In [ ]:
# 4. จัดเก็บบันทึกไฟล์ดัชนีคะแนนประเภทถนนในคลัง interim
output_dir = BASE_DIR / "data" / "interim"
os.makedirs(output_dir, exist_ok=True)

if 'road_traffic_proxy' in locals() and len(road_traffic_proxy) > 0:
    road_output = output_dir / "bangkok_road_traffic_weights.json"
    road_traffic_proxy.to_json(road_output, orient='records', force_ascii=False, indent=4)
    print(f"💾 บันทึกไฟล์ดัชนีค่าน้ำหนักถนนระดับอินเตอร์ริมสำเร็จ: {road_output}")
    
# ล้างแรมระบบ GIS
if 'drive_net' in locals(): del drive_net
if 'local_net' in locals(): del local_net
gc.collect()
print("🧹 ล้างแรมเรียบร้อยปลอดภัยครับ!")